# linear-affine-on-custom-tensor — worked example 3: Bias-optional Linear forward

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `linear-affine-on-custom-tensor`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Real Linear layers support `bias=None`. The forward computes `x @ weight` and only adds the bias (with its Recipe) when one is present. When bias is absent, the matmul output is returned directly as the layer output, so the graph has one node instead of two.

## Worked solution

We make the affine add conditional on whether a bias exists.

1. **Always matmul.** Compute `mm = x @ weight` and attach its Recipe — this step is unconditional.
2. **Branch on bias.** If `bias is None`, return `mm` as the output; the layer is purely linear and the graph is just the matmul node.
3. **Otherwise add bias.** When bias is present, broadcast-add it and attach the second Recipe, exactly as the standard affine map.
4. **Why it matters.** Many architectures disable bias before a normalization layer (the norm's own shift makes it redundant). The forward must handle both, and the recipe chain length differs accordingly.

The demo runs the layer twice — once with a bias, once without — and prints the recipe-chain depth in each case to show the graph shrinks when bias is omitted.

In [ ]:
import numpy as np
from dataclasses import dataclass

np.random.seed(2)

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad
        self.recipe = recipe

def linear_forward(x, weight, bias=None):
    mm = MiniTensor(x.array @ weight.array,
                    requires_grad=(x.requires_grad or weight.requires_grad))
    mm.recipe = Recipe(np.matmul, (x.array, weight.array), {}, {0: x, 1: weight})
    if bias is None:
        return mm
    out = MiniTensor(mm.array + bias.array,
                     requires_grad=(mm.requires_grad or bias.requires_grad))
    out.recipe = Recipe(np.add, (mm.array, bias.array), {}, {0: mm, 1: bias})
    return out

def depth(node):
    d = 0
    while node.recipe is not None and node.recipe.parents:
        d += 1
        node = node.recipe.parents[0]
    return d

x = MiniTensor(np.random.randn(2, 3))
weight = MiniTensor(np.random.randn(3, 4), requires_grad=True)
bias = MiniTensor(np.random.randn(4), requires_grad=True)
print('with bias depth:', depth(linear_forward(x, weight, bias)))
print('no   bias depth:', depth(linear_forward(x, weight, None)))